# Denoising filters: a comparison

Recovering a slow underlying signal from noisy measurements is one of the most
common jobs in a data pipeline. screamer offers several families of filter for
it, and they make different trade-offs: how *smooth* the result is, how much
*lag* (phase delay) it adds, how *robust* it is to outliers, and how much compute
and memory it needs. No single filter wins on all four, so it helps to see them
side by side.

This notebook runs each family on the same noisy signal, a smooth waveform with
broadband noise. Every filter here is *causal*, using only past and present
samples, so its offline and live results are identical (see
[streaming](06-streaming-live-events.ipynb)). The smoothing families share a
window of 31 so they line up. At the end we add outlier spikes and show how a
robust pre-cleaning step lets any of them cope.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import (RollingMean, EwMean, RollingMedian, Butter, MovingAverage,
                      RollingPoly1, RollingPoly2, ImpulseClip)

rng = np.random.default_rng(7)
n = 400
t = np.linspace(0, 6 * np.pi, n)
clean = np.sin(t) + 0.35 * np.sin(2.9 * t + 0.6)   # the signal we want back
noisy = clean + 0.30 * rng.standard_normal(n)      # broadband noise

W = 31   # window for the smoothing families


def show(title, curves, faint=()):
    "Plot the noisy signal and the clean reference, with the given filters on top."
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(noisy, lw=0.5, color="0.75", label="noisy")
    ax.plot(clean, lw=1.1, color="0.35", ls="--", label="clean signal")
    for label, y in curves.items():
        emphasised = label not in faint
        ax.plot(y, lw=1.8 if emphasised else 1.0, alpha=1.0 if emphasised else 0.55, label=label)
    ax.set_title(title)
    ax.set_xlabel("sample")
    ax.set_ylim(-2.2, 2.2)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
    plt.tight_layout()

## Moving average and exponential average

The two workhorses. `RollingMean` averages the last *n* samples with equal
weight; `EwMean` keeps all history but lets it decay geometrically, so recent
samples count more. Both are cheap: `RollingMean` updates in constant time from a
running sum, and `EwMean` holds a single number and needs no window buffer at
all. For a given responsiveness they are the smoothest options here.

The cost is lag and fragility. Equal weighting delays the output by about half the
window; exponential weighting turns with the data sooner for the same nominal
width but still trails. And because every sample enters the average, a lone
outlier would be smeared into the output as a bump lasting a full window (we
handle outliers at the end). When the data trends, a local polynomial fit removes
the lag; both are below.

In [ ]:
show("Moving average and exponential average", {
    "RollingMean(31)":  RollingMean(W)(noisy),
    "EwMean(span=31)":  EwMean(span=W)(noisy),
})

## Median filter

`RollingMedian` reports the middle value of the window instead of the average.
That makes it robust: a few extreme samples cannot move the middle, so it ignores
outliers that pull an average off course, and it keeps step edges sharp where a
mean rounds them off. Neither is something a linear filter can do.

On ordinary noise the mean is hard to beat, because averaging is the best way to
cancel zero-mean noise, so the median below comes out a little rougher than the
mean for the same window. The median earns its place when the data has outliers or
sharp edges, which the final section returns to. The cost is compute and memory:
the window has to be stored and its values kept in order, so an update costs more
than the constant-time running mean.

In [ ]:
show("Median vs mean on ordinary noise", {
    "RollingMedian(31)":  RollingMedian(W)(noisy),
    "RollingMean(31)":    RollingMean(W)(noisy),
})

## Low-pass filters

These are designed in the frequency domain: keep the slow variation, attenuate
the fast wiggles above a cutoff. `Butter` is a Butterworth IIR filter, where a low
order already gives a sharp roll-off and the filter carries only a few state
values however long its memory. `MovingAverage` is an FIR filter you define by its
tap weights, here a raised-cosine (Hann) window, which gives a smooth, stable
response with no feedback.

Low-pass filters give you direct control over which frequencies survive, where the
window statistics set the cutoff only implicitly through their width. The
trade-offs: a Butterworth's phase delay grows with its order and it can overshoot
just after a sharp step, and an FIR needs many taps to make its cutoff steep. Like
the average, both treat a spike as signal and let it through, so pair them with a
robust pre-cleaning step when the data has outliers.

In [ ]:
hann = np.hanning(15)
show("Low-pass filters", {
    "Butter(3, 0.04)":            Butter(3, 0.04)(noisy),
    "MovingAverage(Hann-15)":     MovingAverage((hann / hann.sum()).tolist())(noisy),
})

## Local linear fit

Instead of averaging the window, `RollingPoly1` fits a straight line to it by
least squares and returns the line's value at the most recent sample. Evaluating
the fit at the leading edge, rather than the middle, removes most of the lag a
moving average shows on a trending or ramping signal. The same fit gives the slope
directly with `derivative_order=1` when you want the rate of change.

It costs the window buffer, like any rolling statistic, and it lets a little more
noise through than a plain mean because it extrapolates to the edge instead of
smoothing to the center. On the sloped parts of the signal below it keeps up
without the visible delay of `RollingMean`.

In [ ]:
show("Local linear fit vs moving average", {
    "RollingPoly1(31)":  RollingPoly1(W)(noisy),
    "RollingMean(31)":   RollingMean(W)(noisy),   # shown faint, for contrast
}, faint={"RollingMean(31)"})

## Local quadratic fit

`RollingPoly2` fits a parabola to the window, so it can follow curvature that a
line cannot. Around peaks and troughs, where a moving average flattens the extreme
and a linear fit still cuts the corner, the quadratic holds the height and shape
with very little lag. It is the causal counterpart of a Savitzky-Golay smoother,
and it exposes the curvature too, with `derivative_order=2`.

More flexibility means more noise passes through, so on flat stretches a quadratic
fit is less smooth than the mean or the linear fit. Choose it when preserving the
size and timing of features matters more than maximum smoothness. Overlaid on the
linear fit below, it hugs the peaks the line rounds off.

In [ ]:
show("Local quadratic fit vs local linear fit", {
    "RollingPoly2(31)":  RollingPoly2(W)(noisy),
    "RollingPoly1(31)":  RollingPoly1(W)(noisy),   # shown faint, for contrast
}, faint={"RollingPoly1(31)"})

## Removing outliers before denoising

Real data often carries the occasional spike: a sensor glitch, a bad tick, a
dropped packet. Every filter above except the median treats a spike as signal and
lets it through, so a single outlier leaves a bump that lasts a full window. Adding
a handful of spikes makes this plain.

Remove the spikes first and any of these smoothers can cope. The trick is to
detect a spike by what defines it, a large and isolated jump. In the slowly varying
level a spike is easy to miss, because the signal itself swings by a similar amount
over a window, so a rolling mean and standard deviation set too wide a band. In the
sample-to-sample *change* it stands out sharply: the smooth signal barely moves
between neighbours while a spike leaps and springs back. screamer has this built in
as [`ImpulseClip`](../functions_rolling/ImpulseClip.md): it flags jumps far larger
than the local noise (measured robustly on the first difference) and replaces those
samples with the rolling median. Below, the same three denoisers run on the raw
spiky signal (top) and on the `ImpulseClip`-cleaned signal (bottom).

In [ ]:
spiky = noisy.copy()
hit = rng.choice(n, 14, replace=False)
spiky[hit] += rng.choice([-1.0, 1.0], hit.size) * rng.uniform(2.5, 4.0, hit.size)

cleaned = ImpulseClip(window_size=31, n_sigma=4.0, start_policy="expanding")(spiky)

In [ ]:
fig, (ax_raw, ax_clean) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for ax, source, title in [(ax_raw,   spiky,   "Denoising the raw spiky signal"),
                          (ax_clean, cleaned, "Denoising after despiking")]:
    ax.plot(source, lw=0.5, color="0.8", label="input")
    ax.plot(clean, lw=1.1, color="0.35", ls="--", label="clean signal")
    for name, f in {"RollingMean(31)":  RollingMean(W),
                    "Butter(3, 0.04)":  Butter(3, 0.04),
                    "RollingPoly2(31)": RollingPoly2(W)}.items():
        ax.plot(f(source), lw=1.6, label=name)   # fresh functors per panel
    ax.set_title(title)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(loc="upper right", fontsize=8, ncol=2)
ax_clean.set_xlabel("sample")
plt.tight_layout()

## Choosing a filter

| Family | Strong at | Watch out for |
|---|---|---|
| [`RollingMean`](../functions_rolling/RollingMean.md), [`EwMean`](../functions_ew/EwMean.md) | smoothest result, cheapest to run | lag; one outlier smears across a window |
| [`RollingMedian`](../functions_rolling/RollingMedian.md) | ignores spikes, keeps edges sharp | more compute; rougher than a mean on plain noise |
| [`Butter`](../functions_signal/Butter.md), [`MovingAverage`](../functions_signal/MovingAverage.md) | direct control of the frequency cutoff | phase delay and overshoot; not robust to outliers |
| [`RollingPoly1`](../functions_rolling/RollingPoly1.md) | little lag on trends; gives the slope | passes more noise than a mean |
| [`RollingPoly2`](../functions_rolling/RollingPoly2.md) | preserves peak height and timing; gives curvature | least smooth on flat stretches |

The families also compose. When the data has outliers, removing them first with
[`ImpulseClip`](../functions_rolling/ImpulseClip.md), or a
[`Hampel`](../functions_rolling/Hampel.md) filter, lets any of these smoothers
cope, as the section above shows. For every filter and its parameters, see the
[Filtering](../by_topic/filtering.rst) and [Smoothing](../by_topic/smoothing.rst) sections of the
reference.